# Can Your Nose Predict Your Brain Health?

In 2013-2014, the CDC gave thousands of older Americans (ages 60-80) a scratch-and-sniff smell test *and* a battery of cognitive tests as part of the National Health and Nutrition Examination Survey (NHANES). That means we've got real data linking how well people can identify odors to how they perform on memory, language, and processing speed tasks. Let's poke around and see what we find.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr

sns.set_theme(style='whitegrid', font_scale=1.1)
%matplotlib inline

# Load the data
df = pd.read_csv('nhanes_2013_2014_smell_cognition_60plus.csv')

# Fix near-zero encoding quirk from the original SAS files
numeric_cols = df.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    mask = (df[col].abs() < 1e-10) & df[col].notna()
    df.loc[mask, col] = 0

# Make demographic labels human-readable
df['gender'] = df['RIAGENDR'].map({1: 'Male', 2: 'Female'})
df['race_ethnicity'] = df['RIDRETH3'].map({
    1: 'Mexican American', 2: 'Other Hispanic',
    3: 'Non-Hispanic White', 4: 'Non-Hispanic Black',
    6: 'Non-Hispanic Asian', 7: 'Other/Multi-Racial'
})
df['education'] = df['DMDEDUC2'].map({
    1: '< 9th Grade', 2: '9-11th Grade', 3: 'HS Graduate/GED',
    4: 'Some College/AA', 5: 'College Graduate+'
})

print(f"Loaded {df.shape[0]:,} people and {df.shape[1]} variables.")

In [ ]:
# Let's see what we're working with
cols = ['RIDAGEYR', 'gender', 'education', 'smell_total', 'olfactory_status',
        'cerad_total_learning', 'CFDCSR', 'CFDAST', 'CFDDS']
df[cols].head(10)

In [ ]:
# Who's in the sample?
print(f"Ages {df['RIDAGEYR'].min():.0f} to {df['RIDAGEYR'].max():.0f}, "
      f"average {df['RIDAGEYR'].mean():.0f}")

gender_pct = df['gender'].value_counts(normalize=True) * 100
print(f"{gender_pct['Female']:.0f}% female, {gender_pct['Male']:.0f}% male")

print(f"\nEducation breakdown:")
for level, pct in df['education'].value_counts(normalize=True).sort_index().items():
    print(f"  {level}: {pct*100:.0f}%")

## The Smell Test

Everyone got an 8-item scratch-and-sniff test (the "Pocket Smell Test"). Score 6-8 = normal (normosmia), 4-5 = reduced smell (hyposmia), 0-3 = severely impaired (anosmia). Let's see how people did.

In [ ]:
# Smell score distribution, colored by status
fig, ax = plt.subplots(figsize=(8, 5))

colors = {'Normosmia (6-8)': '#2ecc71', 'Hyposmia (4-5)': '#f39c12',
          'Anosmia (0-3)': '#e74c3c'}

for status in ['Anosmia (0-3)', 'Hyposmia (4-5)', 'Normosmia (6-8)']:
    subset = df[df['olfactory_status'] == status]['smell_total'].dropna()
    ax.hist(subset, bins=np.arange(-0.5, 9.5, 1), alpha=0.7,
            label=f'{status} (n={len(subset)})', color=colors[status],
            edgecolor='white')

ax.set_xlabel('Smell Score (out of 8)')
ax.set_ylabel('Number of People')
ax.set_title('How Well Can Older Adults Identify Smells?')
ax.set_xticks(range(9))
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Cognitive scores at a glance
cog_measures = [
    ('cerad_total_learning', 'Word Learning (CERAD)'),
    ('CFDCSR', 'Delayed Recall'),
    ('CFDAST', 'Animal Naming'),
    ('CFDDS', 'Processing Speed (DSST)'),
]

fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for ax, (col, title) in zip(axes.flat, cog_measures):
    data = df[col].dropna()
    ax.hist(data, bins=30, color='steelblue', edgecolor='white', alpha=0.8)
    ax.axvline(data.mean(), color='red', linestyle='--', linewidth=1.5,
               label=f'Mean = {data.mean():.1f}')
    ax.set_title(title)
    ax.set_xlabel('Score')
    ax.set_ylabel('Count')
    ax.legend(fontsize=9)

plt.suptitle('Four Cognitive Tests', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## Are Smell and Cognition Related?

Here's the big question: do people who score poorly on the smell test also tend to score lower on cognitive tests? Let's start with a correlation heatmap.

In [ ]:
# Correlation heatmap
corr_vars = ['smell_total', 'cerad_total_learning', 'CFDCSR', 'CFDAST',
             'CFDDS', 'RIDAGEYR']
corr_labels = ['Smell Score', 'Word Learning', 'Delayed Recall',
               'Animal Naming', 'Processing Speed', 'Age']

corr_matrix = df[corr_vars].corr()
corr_matrix.index = corr_labels
corr_matrix.columns = corr_labels

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, square=True, ax=ax, mask=mask,
            linewidths=0.5)
ax.set_title('How Everything Correlates')
plt.tight_layout()
plt.show()

### Zooming In: Scatter Plots

Each dot below is a person. The red line shows the overall trend. Notice how every cognitive test tilts upward with higher smell scores.

In [ ]:
# Scatter plots: smell vs each cognitive measure
fig, axes = plt.subplots(2, 2, figsize=(11, 9))

for ax, (col, title) in zip(axes.flat, cog_measures):
    subset = df[['smell_total', col]].dropna()
    x, y = subset['smell_total'], subset[col]

    # Jitter the x-axis since smell scores are integers
    x_jitter = x + np.random.normal(0, 0.12, len(x))
    ax.scatter(x_jitter, y, alpha=0.08, s=15, color='steelblue')

    # Regression line
    z = np.polyfit(x, y, 1)
    x_range = np.linspace(0, 8, 100)
    ax.plot(x_range, np.poly1d(z)(x_range), 'r-', linewidth=2.5)

    # Stats
    r, p = pearsonr(x, y)
    ax.set_title(f'{title}\nr = {r:.3f}, p = {p:.1e}')
    ax.set_xlabel('Smell Score')
    ax.set_ylabel(title)
    ax.set_xticks(range(9))

plt.suptitle('Smell Score vs. Cognitive Performance', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### Group Comparisons

Instead of treating smell as a number, let's split people into three groups (normal, reduced, severely impaired) and compare their cognitive scores side by side.

In [ ]:
# How do cognitive scores look across smell groups?
fig, axes = plt.subplots(2, 2, figsize=(12, 9))

order = ['Normosmia (6-8)', 'Hyposmia (4-5)', 'Anosmia (0-3)']
palette = {'Normosmia (6-8)': '#2ecc71', 'Hyposmia (4-5)': '#f39c12',
           'Anosmia (0-3)': '#e74c3c'}

for ax, (col, title) in zip(axes.flat, cog_measures):
    sns.violinplot(data=df, x='olfactory_status', y=col, order=order,
                   palette=palette, ax=ax, inner='box', cut=0,
                   density_norm='width')
    ax.set_title(title)
    ax.set_xlabel('')
    ax.set_ylabel('Score')

plt.suptitle('Cognitive Scores by Smell Group', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## Which Smells Are Hardest?

Not all 8 odors are equally easy to identify. Some are missed way more often than others — and the ones people struggle with most also tend to be the most linked to cognitive scores.

In [ ]:
# Which odors do people get wrong most often?
odor_items = {
    'CSXCHOOD_correct': 'Chocolate',
    'CSXSBOD_correct': 'Strawberry',
    'CSXSMKOD_correct': 'Smoke',
    'CSXLEAOD_correct': 'Leather',
    'CSXSOAOD_correct': 'Soap',
    'CSXGRAOD_correct': 'Grape',
    'CSXONOD_correct': 'Onion',
    'CSXNGSOD_correct': 'Natural Gas',
}

# Accuracy per odor
accuracy = {name: df[col].mean() * 100 for col, name in odor_items.items()}
accuracy = pd.Series(accuracy).sort_values()

# Mean Processing Speed (DSST) gap: correct vs incorrect identifiers
dsst_gap = {}
for col, name in odor_items.items():
    correct = df[df[col] == 1]['CFDDS'].mean()
    incorrect = df[df[col] == 0]['CFDDS'].mean()
    dsst_gap[name] = correct - incorrect

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: % correct per odor
bars = axes[0].barh(accuracy.index, accuracy.values, color='steelblue',
                    edgecolor='white', height=0.6)
axes[0].set_xlabel('% Correctly Identified')
axes[0].set_title('How Often Each Odor Was Identified Correctly')
for bar, val in zip(bars, accuracy.values):
    axes[0].text(val + 0.5, bar.get_y() + bar.get_height() / 2,
                 f'{val:.0f}%', va='center', fontsize=9)
axes[0].set_xlim(0, 105)

# Right: DSST gap (correct - incorrect) per odor
gap_series = pd.Series(dsst_gap).reindex(accuracy.index)
bar_colors = ['#e74c3c' if name in ('Smoke', 'Natural Gas') else 'steelblue'
              for name in gap_series.index]
axes[1].barh(gap_series.index, gap_series.values, color=bar_colors,
             edgecolor='white', height=0.6)
axes[1].set_xlabel('Processing Speed Gap (correct vs. incorrect)')
axes[1].set_title('How Much Each Odor Predicts Cognitive Score')
for i, (name, val) in enumerate(gap_series.items()):
    axes[1].text(val + 0.3, i, f'+{val:.0f} pts', va='center', fontsize=9)

plt.tight_layout()
plt.show()

print("Red bars = safety-relevant odors (smoke, natural gas)")

## What Did We See?

- **Smell and cognition are linked.** People who score lower on the smell test tend to score lower on *all four* cognitive tests. The correlations aren't huge, but they're consistent.
- **It's not just age.** Age is correlated with both smell and cognition, but the smell-cognition link shows up even when you look within the heatmap alongside age.
- **The pattern is graded.** It's not just "can smell vs. can't smell" -- there's a stepwise drop across normosmia, hyposmia, and anosmia groups.
- **Processing speed (DSST) shows the strongest relationship** with smell, which is interesting because DSST is also one of the most sensitive tests for early cognitive decline.

Want to dig deeper? Check out the full analysis in `nhanes_olfactory_cognition.ipynb`, and see `column_variables.html` for every variable in the dataset.